<a href="https://colab.research.google.com/github/uomna/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uomna/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Logistic Regression, then Random Forest.

My lane is a yes/no question: will a page keep declining? I already have
an observed label (is_declining_label, derived from trend_direction) and
a frozen baseline that flags pages by their CTR-vs-position-tier gap.
Logistic Regression is the natural first learned model here — it's
readable, I can print and interpret its weights, and it lets me compare
fairly against my rule baseline before trying anything more complex.
If the pattern needs more than a straight line, I'll add a Random Forest
next, since features like ctr, position, and engagement may interact.

Base rate: 54.2% of pages are declining (16,262 of 30,000) — a fairly
balanced target, not a rare-event problem.

In [11]:
%cd /content
!rm -rf flyrank-ml
!git clone https://github.com/uomna/flyrank-ml.git
%cd flyrank-ml
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# create the target: 1 if the page is declining, 0 otherwise
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# check the balance of the target
print(df["is_declining_label"].value_counts())
print(df["is_declining_label"].mean())

/content
Cloning into 'flyrank-ml'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 157 (delta 62), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (157/157), 1.87 MiB | 12.87 MiB/s, done.
Resolving deltas: 100% (62/62), done.
/content/flyrank-ml
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
0.5420666666666667


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split: Grouped by client_id, not random and not time-based.

The starter CSV doesn't carry a per-row date, so a time split isn't
available here. A random row split would let the same client appear in
both train and test, so the model could learn "this smells like client X"
instead of learning what a declining page actually looks like. Grouping
by client_id keeps every client on one side only, so the test rows come
from clients the model has never seen. That matches the real deployment
question: will this help on a new client's pages, not just more pages
from clients we already know.

In [12]:
from sklearn.model_selection import GroupShuffleSplit

# use the same rows as the baseline (avg_position must be > 0)
df_valid = df[df["avg_position"] > 0].copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_valid, groups=df_valid["client_id"]))

train_df = df_valid.iloc[train_idx]
test_df = df_valid.iloc[test_idx]

# check no client appears in both sides
overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print("Train rows:", len(train_df), "| Test rows:", len(test_df))
print("Overlapping clients (must be zero):", len(overlap))

Train rows: 22974 | Test rows: 5821
Overlapping clients (must be zero): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Comparison table (precision@50, same test split, same 50 pages):

| method                          | precision@50 |
|----------------------------------|--------------|
| base rate                        | 0.54         |
| baseline (CTR-gap rule)          | 0.56         |
| logistic regression (first try)  | 1.00         |
| logistic regression (final)      | 0.74         |

First attempt scored a suspicious 1.00 precision@50. Checking the model's
coefficients showed impressions_last_30d and impressions_prev_30d had
coefficients 10-30x larger than every other feature — both are almost
certainly the same components trend_pct was built from, so the model was
recovering the label instead of learning a pattern. I removed the whole
last_30d/prev_30d family and retrained.

The honest result: logistic regression reaches 74% precision@50 versus
56% for the rule baseline — an 18-point improvement over the baseline,
and 20 points over guessing (the 54% base rate). This is a real signal,
not a leak.

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# feature columns: numeric signals available before the label,
# excluding trend_direction/trend_pct (label-leaking) and id columns
feature_cols = [
    "search_volume", "competition", "cpc", "word_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

X_train = train_df[feature_cols]
y_train = train_df["is_declining_label"]
X_test = test_df[feature_cols]
y_test = test_df["is_declining_label"]

# fill missing values using the median from the training data only
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

# scale features (Logistic Regression needs this)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

# train the model
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

# get decline probability for each test page
test_df = test_df.copy()
test_df["model_score"] = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Model trained.")
print("Number of features used:", len(feature_cols))
print(test_df[["content_id", "model_score"]].head())

Model trained.
Number of features used: 27
              content_id  model_score
0   content_304f48230142     0.925979
1   content_a1fb4e703a9e     0.999999
5   content_d4084a4bc775     0.909579
6   content_9a34b442b552     0.619293
19  content_af865035b328     0.660376


In [14]:
import numpy as np

# recompute the baseline score exactly as in ML-07, on the full valid set
df_valid["position_bucket"] = pd.cut(
    df_valid["avg_position"],
    bins=[0, 3, 10, 20, float('inf')],
    labels=["1-3", "4-10", "11-20", "21+"]
)
df_valid["tier_avg_ctr"] = df_valid.groupby("position_bucket", observed=True)["ctr"].transform("mean")
df_valid["ctr_gap"] = df_valid["tier_avg_ctr"] - df_valid["ctr"]

visible = (df_valid["impressions_90d"] >= 500).astype(int)
underperforming = (df_valid["ctr_gap"] > 0).astype(int)
df_valid["baseline_score"] = underperforming * visible * df_valid["ctr_gap"] * df_valid["impressions_90d"]

# bring the baseline score into the test set only, matched by content_id
test_df = test_df.merge(
    df_valid[["content_id", "baseline_score"]], on="content_id", how="left"
)

# precision@50 for each ranking, computed on the same test rows
def precision_at_k(frame, score_col, k=50):
    top_k = frame.sort_values(score_col, ascending=False).head(k)
    return top_k["is_declining_label"].mean()

base_rate = test_df["is_declining_label"].mean()
baseline_p50 = precision_at_k(test_df, "baseline_score", 50)
model_p50 = precision_at_k(test_df, "model_score", 50)

comparison = pd.DataFrame({
    "method": ["base_rate", "baseline (CTR-gap rule)", "logistic_regression"],
    "precision_at_50": [base_rate, baseline_p50, model_p50]
})
print(comparison)

                    method  precision_at_50
0                base_rate         0.539942
1  baseline (CTR-gap rule)         0.560000
2      logistic_regression         1.000000


In [15]:
import numpy as np

coef_table = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": log_reg.coef_[0]
})
coef_table["abs_coefficient"] = coef_table["coefficient"].abs()
coef_table = coef_table.sort_values("abs_coefficient", ascending=False)
print(coef_table.head(10))

                 feature  coefficient  abs_coefficient
14  impressions_last_30d   -35.538588        35.538588
17  impressions_prev_30d    29.318407        29.318407
4        impressions_90d     1.637019         1.637019
15       clicks_last_30d    -1.173860         1.173860
18       clicks_prev_30d     1.064591         1.064591
8              users_90d    -0.852998         0.852998
7           sessions_90d     0.848755         0.848755
6          pageviews_90d     0.464550         0.464550
16     sessions_last_30d    -0.399372         0.399372
20      content_age_days    -0.323397         0.323397


In [16]:
# remove the last_30d / prev_30d family — too close to how the label was built
leaky_cols = [
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"
]
feature_cols_clean = [c for c in feature_cols if c not in leaky_cols]

X_train_c = train_df[feature_cols_clean]
X_test_c = test_df[feature_cols_clean]

imputer_c = SimpleImputer(strategy="median")
X_train_c_imp = imputer_c.fit_transform(X_train_c)
X_test_c_imp = imputer_c.transform(X_test_c)

scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c_imp)
X_test_c_scaled = scaler_c.transform(X_test_c_imp)

log_reg_clean = LogisticRegression(max_iter=1000, random_state=42)
log_reg_clean.fit(X_train_c_scaled, y_train)

test_df["model_score_clean"] = log_reg_clean.predict_proba(X_test_c_scaled)[:, 1]

model_clean_p50 = precision_at_k(test_df, "model_score_clean", 50)

comparison_v2 = pd.DataFrame({
    "method": ["base_rate", "baseline (CTR-gap rule)", "logistic_regression (leaky)", "logistic_regression (clean)"],
    "precision_at_50": [base_rate, baseline_p50, model_p50, model_clean_p50]
})
print(comparison_v2)

                        method  precision_at_50
0                    base_rate         0.539942
1      baseline (CTR-gap rule)         0.560000
2  logistic_regression (leaky)         1.000000
3  logistic_regression (clean)         0.740000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

What the model leans on:

Top features by coefficient size: users_90d, sessions_90d,
content_age_days, days_with_impressions, avg_position, ctr. These are
plausible — pages with fewer engaged users/sessions and lower CTR
relative to position are reasonable signals of decline. No single
feature dominates the way the leaky ones did, which is a healthy sign.

3 wrong cases (model said "will decline", but the page actually improved):

1. content_c84a0ab98e90 — model_score 0.86, ctr 0.03% at position 7.8,
   223K impressions. Actual: stable, trend_pct +17.2%. The model likely
   over-weighted the very low CTR, without accounting for the page's
   huge impression volume, which may reflect a broad/branded query where
   low CTR is normal rather than a warning sign.

2. content_26d48a980581 — model_score 0.84, ctr 0.00% at position 4.6.
   Actual: trending up, trend_pct +84.8%. A CTR of exactly zero with a
   good position looks alarming, but with only 1,266 impressions this
   is a small, noisy sample — one or two extra clicks would swing the
   rate a lot. The model has no way to see that this number is unstable.

3. content_c94a53e3bfb8 — model_score 0.82, ctr 0.23% at position 8.1.
   Actual: trending up, trend_pct +71.0%. Similar pattern: a low CTR at
   a mid-tier position, but the page was actually gaining momentum. The
   model doesn't have access to a momentum feature (which we removed as
   leaky), so it can't distinguish "consistently weak" from "recovering."

Common thread: the model treats low CTR-at-position as a static warning
sign, but it can't tell a stable weak page apart from one that is
already turning around, especially on low-impression pages where CTR is
noisy. A useful next step would be a stability flag (e.g. minimum
impression threshold) rather than reading CTR alone.

In [17]:
coef_table_clean = pd.DataFrame({
    "feature": feature_cols_clean,
    "coefficient": log_reg_clean.coef_[0]
})
coef_table_clean["abs_coefficient"] = coef_table_clean["coefficient"].abs()
coef_table_clean = coef_table_clean.sort_values("abs_coefficient", ascending=False)
print(coef_table_clean.head(10))

                  feature  coefficient  abs_coefficient
8               users_90d    -1.165402         1.165402
7            sessions_90d     0.958395         0.958395
14       content_age_days    -0.387111         0.387111
12  days_with_impressions     0.379026         0.379026
13     days_with_sessions    -0.365935         0.365935
11      scroll_events_90d     0.361378         0.361378
17           avg_position    -0.196103         0.196103
5              clicks_90d    -0.184882         0.184882
6           pageviews_90d     0.182019         0.182019
16                    ctr    -0.137622         0.137622


In [18]:
top50_model = test_df.sort_values("model_score_clean", ascending=False).head(50)

# wrong cases: model ranked them high, but they did NOT decline
wrong_cases = top50_model[top50_model["is_declining_label"] == 0]

print("Number of wrong cases in top 50:", len(wrong_cases))
print(wrong_cases[["content_id", "model_score_clean", "ctr", "avg_position",
                    "impressions_90d", "trend_direction", "trend_pct"]].head(3))

Number of wrong cases in top 50: 13
                content_id  model_score_clean   ctr  avg_position  \
1369  content_c84a0ab98e90           0.860667  0.03           7.8   
5422  content_26d48a980581           0.836134  0.00           4.6   
1601  content_c94a53e3bfb8           0.824680  0.23           8.1   

      impressions_90d trend_direction  trend_pct  
1369           223271          stable       17.2  
5422             1266              up       84.8  
1601             2164              up       71.0  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.